# Creating Custom Layers and Models
- Create custom layers and integrate them into a keras model
- Compile, train and evaultare the model

In [3]:
#### install & import libraries
%pip install tensorflow==2.20.0
%pip install pydot graphviz

  Using cached tensorflow-2.20.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached tensorflow-2.20.0-cp313-cp313-macosx_12_0_arm64.whl (200.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 10.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import Sequential

### Define a custom layer
Define a custom layer with 32 units and ReLU activation

In [10]:
class CustomDenseLayer(Layer):
    def __init__(self, units=32):
        super(CustomDenseLayer, self).__init__()
        self.units = units

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer='random_normal',
                                 trainable=True)
        self.b = self.add_weight(shape=(self.units,),
                                 initializer='zeros',
                                 trainable=True)
    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

### Integrate the custom layer into a model
Create a Keras model using the custom layer

In [11]:
from tensorflow.keras.layers import Softmax
model = Sequential([
    CustomDenseLayer(128),
    CustomDenseLayer(10),
    Softmax()
])

The **Softmax** activation function is used in the output layer for multi-class classification tasks, ensuring the model outputs probabilities that sum to 1 for each class, which aligns with the categorical cross-entropy loss function. This adjustment ensures the model is optimized correctly for multiclass classification

### Compile the model
Compile the model with the Adam optimizer and categorical cross-entropy loss

In [12]:
model.compile(optimizer='adam', loss='categorical_crossentropy')
print("Model summary before building:")
model.summary()

model.build((1000, 20))
print("\nModel summary after building:")
model.summary()

Model summary before building:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ custom_dense_layer_4            │ ?                      │   0 (unbuilt) │
│ (CustomDenseLayer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_dense_layer_5            │ ?                      │   0 (unbuilt) │
│ (CustomDenseLayer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_2 (Softmax)             │ ?                      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ custom_dense_layer_4            │ (1000, 128)            │         2,688 │
│ (CustomDenseLayer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_dense_layer_5            │ (1000, 10)             │         1,290 │
│ (CustomDenseLayer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_2 (Softmax)             │ (1000, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,978 (15.54 KB)

 Trainable params: 3,978 (15.54 KB)

 Non-trainable params: 0 (0.00 B)

### Train the model
Train the model on some example data, generate random data for training.

In [13]:
import numpy as np

x_train = np.random.random((1000, 20))
y_train = np.random.randint(10, size=(1000, 1))

# convert labels to categorical one-hot encoding
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
model.fit(x_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 763us/step - loss: 2.3020
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.2985
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 698us/step - loss: 2.2971
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step - loss: 2.2962
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 660us/step - loss: 2.2940
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 716us/step - loss: 2.2919
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step - loss: 2.2911
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 662us/step - loss: 2.2885
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 679us/step - loss: 2.2857
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 679us/step - loss: 2.2836


### Evaluate the model

Evaluate the model using test data to see its performance. 

In [18]:
# Generate random test data 
x_test = np.random.random((200, 20)) 
y_test = np.random.randint(10, size=(200, 1)) 

# Convert labels to categorical one-hot encoding 
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10) 

# Evaluate the model 
loss = model.evaluate(x_test, y_test) 
print(f'Test loss: {loss}') 

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.3152 
Test loss: 2.315173864364624


In [19]:
from tensorflow.keras.utils import plot_model

# visualize the model
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [20]:
# Add dropout layer
from tensorflow.keras.layers import Dropout

# Modify the model to include a Dropout layer
model = Sequential([
    CustomDenseLayer(64),
    Dropout(0.5),
    CustomDenseLayer(10)
])

# Recompile the model
model.compile(optimizer='adam', loss='categorical_crossentropy')

# Train the model again
model.fit(x_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 735us/step - loss: 6.8172 
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 772us/step - loss: 3.7111
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step - loss: 2.6421
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 653us/step - loss: 2.3647
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 672us/step - loss: 2.3545
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 703us/step - loss: 2.3659
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 748us/step - loss: 2.3197
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 724us/step - loss: 2.3182
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 742us/step - loss: 2.3027
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 729us/step - loss: 2.3013
